# Petit réseau, dropout Montmory et Threefry sur TPU

Prototype local validé sur CPU ; compilation et performances natives TPU à vérifier. Deux couches cachées de 128 neurones, 21 252 paramètres, données synthétiques, dropout à 25 %. Trois graines, 150 étapes par variante. Choisir **Exécution > Modifier le type d’environnement > TPU**, puis exécuter les cellules dans l’ordre. Aucun accès à votre compte GitHub ni envoi automatique de résultats.

Le contrat exact porte sur les entiers et les masques ; l’entraînement flottant n’est pas garanti bit à bit entre architectures.

## 1. Vérifier le TPU
Cette cellule installe JAX 0.10.2 et libtpu si nécessaire, puis refuse tout repli CPU. Elle affiche et conserve la sortie complète du sous-processus. En cas d’échec, passer directement à la cellule 4 pour télécharger le diagnostic. Si aucun TPU n’est alloué, reconnecter un environnement TPU avant de continuer ; un quota Colab exige d’attendre le rétablissement de l’accès ou de revoir l’offre de calcul.

In [ ]:
import datetime, importlib.metadata, os, subprocess, sys
from pathlib import Path
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
TPU_PREFLIGHT_LOG = Path.cwd() / ("montmory-tpu-preflight-" + stamp + ".log")
TPU_PREFLIGHT_OK = False
TPU_PREFLIGHT_LOG.write_text("Python: " + sys.version + "\n")
print("Diagnostic log:", TPU_PREFLIGHT_LOG, flush=True)

def run_logged(command, env=None):
    with TPU_PREFLIGHT_LOG.open("a") as log:
        with subprocess.Popen(command, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                code = process.wait()
            except BaseException:
                process.terminate()
                process.wait()
                raise
        log.write("Exit status: " + str(code) + "\n")
    if code:
        raise RuntimeError("Setup / TPU access check failed (exit " + str(code)
                           + "). See the output above. Diagnostic log: " + str(TPU_PREFLIGHT_LOG)
                           + ". Use cell 4 to download it. No benchmark was started.")

try:
    ready = (importlib.metadata.version("jax") == "0.10.2"
             and importlib.metadata.version("jaxlib") == "0.10.2"
             and bool(importlib.metadata.version("libtpu")))
except importlib.metadata.PackageNotFoundError:
    ready = False
if not ready:
    run_logged([sys.executable, "-m", "pip", "install", "--upgrade", "jax[tpu]==0.10.2"])
probe = "import jax; assert jax.default_backend() == 'tpu'; print(jax.__version__, jax.devices())"
run_logged([sys.executable, "-u", "-c", probe], env={**os.environ, "JAX_PLATFORMS": "tpu"})
TPU_PREFLIGHT_OK = True


## 2. Extraire les sources contrôlées
Le kit est intégré à ce notebook ; aucune publication GitHub n’est nécessaire.

In [ ]:
import base64, hashlib, io, json, tarfile
from pathlib import Path
import datetime
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == 'e71d3d504302e9b768aa9153721da5b814f9d8ab708247c76da0a307b6cb27a7'
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
KIT = Path("/content") / ("montmory-dropout-" + stamp)
KIT.mkdir(exist_ok=False)
with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as tar:
    for member in tar.getmembers():
        target = (KIT / member.name).resolve()
        if not member.isfile() or not target.is_relative_to(KIT.resolve()):
            raise ValueError("Unexpected archive member: " + member.name)
        target.parent.mkdir(parents=True, exist_ok=True)
        with tar.extractfile(member) as source:
            target.write_bytes(source.read())
manifest = json.loads((KIT / "manifest.json").read_text())
for rel, expected in manifest.items():
    assert hashlib.sha256((KIT / rel).read_bytes()).hexdigest() == expected, rel
print("Source hashes PASS:", len(manifest), "files")


## 3. Contrôler et entraîner
Les compilations précèdent les mesures. Prévoir plusieurs minutes, selon l’allocation et la compilation. Les quatre chemins sont JAX Threefry, JAX Montmory, Pallas Threefry et Pallas Montmory. Les masques Pallas doivent correspondre à leur propre référence. Un échec de compilation TPU reste possible à ce stade du prototype.

In [ ]:
import subprocess, sys
if not globals().get("TPU_PREFLIGHT_OK", False):
    raise RuntimeError("Run cell 1 successfully before starting the TPU benchmark.")
RESULTS = KIT / "results" / "dropout-tpu"
RESULT_ARCHIVE = RESULTS.with_suffix(".tar.gz")
command = [sys.executable, "-u", str(KIT / "applications/dropout/run.py"),
           "--backend", "tpu", "--out", str(RESULTS), "--steps", "150"]
print("Checks, compilation, then 150 updates per seed and implementation.", flush=True)
with (KIT / "run.log").open("w") as log:
    with subprocess.Popen(command, cwd=KIT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
print("Archive:", RESULT_ARCHIVE)
if code:
    raise RuntimeError("Run failed. Use the next cell to download diagnostics.")


## 4. Télécharger résultats ou diagnostics
Exécuter aussi après une erreur à l’étape 1 ou 3. Si la vérification initiale échoue, cette cellule télécharge son journal même sans kit extrait. Aucun chiffre de vitesse CPU n’est présenté comme une mesure TPU.

In [ ]:
import tarfile
from google.colab import files
# Cell 1 can fail before KIT / RESULTS exist.
preflight = globals().get("TPU_PREFLIGHT_LOG")
results = globals().get("RESULTS")
kit = globals().get("KIT")
if not globals().get("TPU_PREFLIGHT_OK", False) and preflight is not None and preflight.exists():
    files.download(str(preflight))
elif results is not None and results.exists():
    archive = results.with_suffix(".tar.gz")
    with tarfile.open(archive, "w:gz") as tar:
        tar.add(results, arcname=results.name, filter=lambda i: None if i.name.endswith((".so", ".dylib")) else i)
        for name in ("run.log", "manifest.json"):
            if kit is not None and (kit / name).exists():
                tar.add(kit / name, arcname=name)
        if preflight is not None and preflight.exists():
            tar.add(preflight, arcname=preflight.name)
    files.download(str(archive))
elif preflight is not None and preflight.exists():
    files.download(str(preflight))
else:
    raise RuntimeError("No diagnostics found. Run cell 1 first.")
